# 20-Domain Retention Campaign

**BASE vs ANCHOR generation across 20 diverse domains**

Тестируем anchor bias на удержание семантических ограничений при свободной генерации.

Домены: dietary (vegan, halal, gluten-free), code (FastAPI, Rust, PostgreSQL, TypeScript, K8s),
math (proof), legal (GDPR), medical (drug-free pain), academic style, zero-waste, organic farming,
budget travel, minimalist design, functional programming, metric units, Python typed.

---
Runtime: **GPU (T4 / A100 / L4)**  
Время: ~30-60 мин на все 20 доменов  
Модель: Qwen3.5-4B (та же что в geometry probe)

## 0. GPU Check

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('GPU not found — Runtime -> Change runtime type -> GPU')

## 1. Clone repo

In [ ]:
import os

REPO_URL = 'https://github.com/kharkilirov1/Anchor-engine.git'
BRANCH = 'claude/review-project-commits-IJRAf'
REPO_DIR = '/content/ABPT'

if os.path.exists(REPO_DIR):
    print('Repo exists -> fetch + checkout')
    !cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    print(f'Cloning branch {BRANCH}...')
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')
!git log --oneline -5

## 2. Install dependencies

In [ ]:
%%time
!pip install -q --upgrade pip
!pip install -q torch torchvision accelerate einops scipy numpy pytest
!pip install -q "transformers @ git+https://github.com/huggingface/transformers.git@main"
print('Dependencies installed')

## 3. Quick test

In [ ]:
!python -m pytest tests/test_qwen_generation_bias.py -v --tb=short

## 4. Preview: list all 20 domains

In [ ]:
import sys
sys.path.insert(0, '.')
from src.data.retention_domains import RETENTION_DOMAINS

print(f'Total domains: {len(RETENTION_DOMAINS)}')
print()
print(f'{"#":>2} {"Domain":<35} {"Profile":<14} {"+ kw":>5} {"- kw":>5} {"Tokens":>6}')
print('-' * 75)
for i, d in enumerate(RETENTION_DOMAINS, 1):
    print(f'{i:>2} {d.name:<35} {d.bias_profile_name:<14} {len(d.positive_keywords):>5} {len(d.negative_keywords):>5} {d.max_new_tokens:>6}')

## 5. Run full campaign (all 20 domains)

Модель загружается один раз, затем прогоняет BASE и ANCHOR генерацию для каждого домена.

In [ ]:
%%time
# Full campaign — all 20 domains on Qwen3.5-4B (same model as geometry probe)
!python scripts/run_qwen_20domain_retention_campaign.py \
    --model Qwen/Qwen3.5-4B \
    --max_new_tokens 500 \
    --device cuda

## 6. Results: summary table

In [ ]:
import json
from pathlib import Path

campaign_dir = Path('archive/20domain_campaign')
json_files = sorted(campaign_dir.glob('campaign_*.json'),
                     key=lambda p: p.stat().st_mtime, reverse=True)

if not json_files:
    print('No results yet. Run the campaign first (cell above).')
else:
    data = json.loads(json_files[0].read_text())
    s = data['summary']
    
    print(f"Model: {data['model']}")
    print(f"Domains: {data['n_domains']}")
    print(f"Total time: {data['total_time_s']:.0f}s")
    print()
    print(f"  ANCHOR wins: {s['anchor_wins']}/{data['n_domains']}")
    print(f"  BASE wins:   {s['base_wins']}/{data['n_domains']}")
    print(f"  Ties:        {s['ties']}/{data['n_domains']}")
    print(f"  Identical:   {s['identical']}/{data['n_domains']}")
    print(f"  Mean delta:  {s['mean_delta_quality']:+.3f}")
    print()
    
    print(f'{"#":>2} {"Domain":<35} {"Profile":<12} {"Base Q":>7} {"Anch Q":>7} {"Delta":>7} {"Result":<10}')
    print('-' * 85)
    for i, r in enumerate(data['results'], 1):
        if 'error' in r:
            print(f"{i:>2} {r['domain']:<35} {r.get('bias_profile','?'):<12} {'ERROR':>7} {'':>7} {'':>7} {r['error'][:20]}")
            continue
        tag = 'WIN' if r['anchor_wins'] else ('LOSS' if r['delta_quality'] < 0 else 'TIE')
        if r.get('identical'):
            tag = 'IDENTICAL'
        bq = r['base_analysis']['quality_score']
        aq = r['anchor_analysis']['quality_score']
        print(f"{i:>2} {r['domain']:<35} {r['bias_profile']:<12} {bq:>7.1f} {aq:>7.1f} {r['delta_quality']:>+7.1f} {tag:<10}")

## 7. Results: keyword details per domain

In [ ]:
import json
from pathlib import Path

campaign_dir = Path('archive/20domain_campaign')
json_files = sorted(campaign_dir.glob('campaign_*.json'),
                     key=lambda p: p.stat().st_mtime, reverse=True)

if not json_files:
    print('No results yet.')
else:
    data = json.loads(json_files[0].read_text())
    
    for r in data['results']:
        if 'error' in r:
            continue
        ba = r['base_analysis']
        aa = r['anchor_analysis']
        tag = 'WIN' if r['anchor_wins'] else ('LOSS' if r['delta_quality'] < 0 else 'TIE')
        
        print(f"\n{'='*60}")
        print(f"  {r['domain']} [{tag}]  (profile: {r['bias_profile']})")
        print(f"{'='*60}")
        print(f"  BASE:   +{ba['positive_total']} / -{ba['negative_total']}  degen={ba['degeneracy_penalty']:.1f}  Q={ba['quality_score']:.1f}")
        print(f"  ANCHOR: +{aa['positive_total']} / -{aa['negative_total']}  degen={aa['degeneracy_penalty']:.1f}  Q={aa['quality_score']:.1f}")
        print(f"  Bias active: {r['active_bias_steps']}/{r['anchor_total_steps']} steps")
        
        if ba.get('positive_hits'):
            top_pos = sorted(ba['positive_hits'].items(), key=lambda x: -x[1])[:5]
            print(f"  BASE top+: {', '.join(f'{k}({v})' for k,v in top_pos)}")
        if ba.get('negative_hits'):
            top_neg = sorted(ba['negative_hits'].items(), key=lambda x: -x[1])[:5]
            print(f"  BASE top-: {', '.join(f'{k}({v})' for k,v in top_neg)}")
        if aa.get('positive_hits'):
            top_pos = sorted(aa['positive_hits'].items(), key=lambda x: -x[1])[:5]
            print(f"  ANCH top+: {', '.join(f'{k}({v})' for k,v in top_pos)}")
        if aa.get('negative_hits'):
            top_neg = sorted(aa['negative_hits'].items(), key=lambda x: -x[1])[:5]
            print(f"  ANCH top-: {', '.join(f'{k}({v})' for k,v in top_neg)}")

## 8. Results: sample generations (first 500 chars)

In [ ]:
import json
from pathlib import Path

campaign_dir = Path('archive/20domain_campaign')
json_files = sorted(campaign_dir.glob('campaign_*.json'),
                     key=lambda p: p.stat().st_mtime, reverse=True)

if not json_files:
    print('No results yet.')
else:
    data = json.loads(json_files[0].read_text())
    
    # Show specific domains of interest
    show_domains = ['vegan_meal_plan', 'fastapi_service', 'halal_cuisine',
                    'rust_safe_only', 'gdpr_data_policy', 'drug_free_pain_management']
    
    for r in data['results']:
        if r['domain'] not in show_domains or 'error' in r:
            continue
        tag = 'WIN' if r['anchor_wins'] else ('LOSS' if r['delta_quality'] < 0 else 'TIE')
        print(f"\n{'#'*70}")
        print(f"  {r['domain']} [{tag}]")
        print(f"{'#'*70}")
        print(f"\n--- BASE (first 500 chars) ---")
        print(r.get('base_continuation', '')[:500])
        print(f"\n--- ANCHOR (first 500 chars) ---")
        print(r.get('anchor_continuation', '')[:500])

## 9. Analysis: win rate by profile type

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

campaign_dir = Path('archive/20domain_campaign')
json_files = sorted(campaign_dir.glob('campaign_*.json'),
                     key=lambda p: p.stat().st_mtime, reverse=True)

if not json_files:
    print('No results yet.')
else:
    data = json.loads(json_files[0].read_text())
    
    by_profile = defaultdict(lambda: {'wins': 0, 'losses': 0, 'ties': 0, 'total': 0, 'deltas': []})
    
    for r in data['results']:
        if 'error' in r:
            continue
        p = r['bias_profile']
        by_profile[p]['total'] += 1
        by_profile[p]['deltas'].append(r['delta_quality'])
        if r['anchor_wins']:
            by_profile[p]['wins'] += 1
        elif r['delta_quality'] < 0:
            by_profile[p]['losses'] += 1
        else:
            by_profile[p]['ties'] += 1
    
    print(f'{"Profile":<14} {"Total":>5} {"Wins":>5} {"Losses":>6} {"Ties":>5} {"Win%":>6} {"Mean Δ":>8}')
    print('-' * 55)
    for p, stats in sorted(by_profile.items()):
        win_pct = 100.0 * stats['wins'] / max(stats['total'], 1)
        mean_d = sum(stats['deltas']) / max(len(stats['deltas']), 1)
        print(f"{p:<14} {stats['total']:>5} {stats['wins']:>5} {stats['losses']:>6} {stats['ties']:>5} {win_pct:>5.0f}% {mean_d:>+8.2f}")
    
    print()
    total_wins = sum(s['wins'] for s in by_profile.values())
    total_all = sum(s['total'] for s in by_profile.values())
    all_deltas = [d for s in by_profile.values() for d in s['deltas']]
    print(f"{'TOTAL':<14} {total_all:>5} {total_wins:>5} {'':>6} {'':>5} {100*total_wins/max(total_all,1):>5.0f}% {sum(all_deltas)/max(len(all_deltas),1):>+8.2f}")

## 10. Optional: run subset of domains

Если полная кампания слишком долгая, можно запустить только интересные домены:

In [ ]:
# Uncomment and run specific domains:
# !python scripts/run_qwen_20domain_retention_campaign.py \
#     --model Qwen/Qwen3.5-4B \
#     --domains vegan_meal_plan,halal_cuisine,gluten_free_bakery,drug_free_pain_management \
#     --max_new_tokens 300 \
#     --device cuda

## 11. Download results

In [ ]:
import shutil, os
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
out_name = f'abpt_20domain_campaign_{timestamp}'
out_dir = f'/tmp/{out_name}'
os.makedirs(out_dir, exist_ok=True)

# Copy campaign results
campaign_src = 'archive/20domain_campaign'
if os.path.exists(campaign_src):
    shutil.copytree(campaign_src, f'{out_dir}/20domain_campaign', dirs_exist_ok=True)

# Copy key state files
for f in ['playbook.md', 'research_state.json']:
    if os.path.exists(f):
        shutil.copy(f, out_dir)

shutil.make_archive(f'/tmp/{out_name}', 'zip', '/tmp', out_name)

try:
    from google.colab import files
    files.download(f'/tmp/{out_name}.zip')
    print(f'Downloaded: {out_name}.zip')
except ImportError:
    print(f'Saved: /tmp/{out_name}.zip')

---

## Cheatsheet

| Profile | Domains | alpha | Penalty |
|---------|---------|-------|---------|
| vegan | vegan_meal_plan | 0.32 | 8.0 |
| dietary | halal_cuisine, gluten_free_bakery | 0.40 | 8.0 |
| code | fastapi, kubernetes, python_typed | 0.90 | 6.0 |
| code_strict | rust_safe, postgresql, typescript, functional | 0.85 | 7.0 |
| math | proof_by_contradiction | 0.60 | 4.0 |
| legal | gdpr_data_policy | 0.55 | 5.0 |
| medical | drug_free_pain_management | 0.50 | 6.0 |
| constraint | metric_units, renewable_energy, formal_academic, zero_waste, organic_farming, budget_travel, minimalist_ui | 0.65 | 5.0 |

**Key metric:** `quality_score = lexical_score - degeneracy_penalty - 1.5 * negative_hits`

**Win condition:** `anchor_quality_score > base_quality_score`